# IT9002 Lab 1.1: Getting Started

**Unit 01: Language, Text, and the NLP Pipeline**

**IT9002 Natural Language Processing**

## Objectives

By the end of this lab, the following should be clear:

- Variables, strings, lists and loops in a notebook, and how values carry from one cell to the next
- Loading a data file from Moodle into the notebook and inspecting it with pandas
- Installing and importing NLTK and spaCy with their resources, and checking the versions in use
- Saving the notebook and its outputs, and reading an error message

## Data

- **SMS Spam Collection** (Almeida, Hidalgo and Yamakami, 2011), UCI Machine Learning Repository dataset 228, https://archive.ics.uci.edu/dataset/228/sms+spam+collection, license CC BY 4.0.

- The course file `sms_spam.csv` (473 KB, on Moodle).

## 1. Python in the Notebook

A code cell executes Python code and displays its output below the cell. A variable remains available to other cells after it has been defined, given the runtime has not been restarted.

The cell below defines two variables and prints their values. The `f` before the quotation mark indicates an **f-string**. When the string is evaluated, each expression inside braces is replaced by its value.

In [1]:
course = "IT9002"
unit = 1
print(f"{course}, Unit {unit}")

IT9002, Unit 1


The cell below divides a message into words. `split()` divides the string at whitespace and returns a list of words. `len()` returns the number of items in the list, while `[:5]` selects the first 5 items.

In [2]:
message = "Free entry in 2 a wkly comp to win FA Cup final tkts"
words = message.split()
print(len(words), "words")
print(words[:5])

13 words
['Free', 'entry', 'in', '2', 'a']


A `for` loop executes its indented block once for each item in a sequence. In the cell below, the loop prints each word together with its length and lowercase form.

In [3]:
for word in words[:5]:
    print(word, len(word), word.lower())

Free 4 free
entry 5 entry
in 2 in
2 1 2
a 1 a


## 2. Loading and Inspecting a File

Each lab uses a small data file that can be downloaded from Moodle. Download `sms_spam.csv` to your computer, then run the cell below. If the file is already available in the runtime, the cell loads it directly; otherwise, it opens a file-upload dialog.

`pd.read_csv()` reads the file into a **DataFrame**, which is a table containing named columns. `df.shape` returns the number of rows and columns in the DataFrame.

In [4]:
from pathlib import Path
import pandas as pd

DATA = "sms_spam.csv"
path = next((p for p in (Path(DATA), Path("data") / DATA) if p.exists()), None)
if path is None:                                   
    from google.colab import files
    files.upload()
    path = Path(DATA)

df = pd.read_csv(path)
print(df.shape)

(5572, 2)


`head()` displays the first 5 rows of the DataFrame.

In [5]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


`info()` summarizes the columns, their data types, and the number of non-null values in each column. If a column has fewer non-null values than the total number of rows, it contains **missing** values.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 541.4 KB


Square brackets containing a column name select that column. `value_counts()` counts the occurrences of each distinct value in the selected column.

In [7]:
df["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

`df.loc[101]` selects the row with index label 101.

In [8]:
df.loc[101]

label                                                    ham
message    I'm still looking for a car to buy. And have n...
Name: 101, dtype: str

`.str.len()` returns the length of each message in characters. `describe()` then summarizes these lengths using the count, mean, standard deviation, minimum, quartiles, and maximum.

In [9]:
df["message"].str.len().describe()

count    5572.000000
mean       80.118808
std        59.690841
min         2.000000
25%        36.000000
50%        61.000000
75%       121.000000
max       910.000000
Name: message, dtype: float64

## 3. Setting Up the Libraries

The labs use the following NLP libraries:

- **NLTK** provides a broad collection of tools and resources for working with text. It supports tasks such as tokenization, stemming, lemmatization, part-of-speech tagging, and access to linguistic datasets and word lists.

- **spaCy** provides efficient, trained language-processing pipelines for analyzing text. It supports tasks such as tokenization, part-of-speech tagging, lemmatization, syntactic analysis, and identifying named entities.

Both libraries are preinstalled in Google Colab, so the cell below does not install them there. In other environments, the libraries will be installed if they are not already available.

In [10]:
import importlib.util, subprocess, sys

for pkg in ("nltk", "spacy"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("libraries are available")

libraries are available


NLTK provides some of its models and word lists as separate downloadable resources. The cell below downloads the **4** resources (~17 MB) that will be used in the labs. 

Downloading the complete NLTK collection of resources with `nltk.download('all')` would require approximately 811 MB to download and 3.7 GB of disk space, so it is unnecessary for these labs.

In [11]:
import nltk

for resource in ("punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"):
    nltk.download(resource, quiet=True)
print("NLTK resources are ready")

NLTK resources are ready


**spaCy** organizes its language-processing capabilities into named pipelines. For English, `en_core_web_sm` is a small pretrained pipeline, approximately 15 MB in size, and is already available in Google Colab.

The cell below:

- Loads the `en_core_web_sm` pipeline, and downloads it first if it is not available.
- Prints the versions of pandas, NLTK, spaCy, and the loaded spaCy pipeline.

Recording version information makes it easier to identify the software environment in which results were produced. This is useful because Google Colab updates its libraries periodically, and newer versions may produce slightly different outputs.

In [12]:
import spacy

try:
    nlp_spacy = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp_spacy = spacy.load("en_core_web_sm")

print("pandas", pd.__version__, "| NLTK", nltk.__version__, "| spaCy", spacy.__version__,
      "| model", nlp_spacy.meta["name"], nlp_spacy.meta["version"])

pandas 3.0.2 | NLTK 3.9.4 | spaCy 3.8.16 | model core_web_sm 3.8.0


The cell below applies both libraries to the message `101`. Each library divides the message into **tokens**, which correspond to the individual units produced by the tokenization process.

In [13]:
from nltk.tokenize import word_tokenize

message = df.loc[101, "message"]
print(word_tokenize(message))
print([token.text for token in nlp_spacy(message)])

['I', "'m", 'still', 'looking', 'for', 'a', 'car', 'to', 'buy', '.', 'And', 'have', 'not', 'gone', '4the', 'driving', 'test', 'yet', '.']
['I', "'m", 'still', 'looking', 'for', 'a', 'car', 'to', 'buy', '.', 'And', 'have', 'not', 'gone', '4the', 'driving', 'test', 'yet', '.']


Both libraries separate punctuation from the surrounding words: the token lists are identical for this particular message, although they may differ for other text.

## 4. Saving the Work

Google Colab saves a notebook and its outputs to the **Google Drive** account of the signed-in user. To save a local copy for distribution, use **File > Download > Download .ipynb**.

Files uploaded to the runtime are temporary and are not stored inside the notebook. If the runtime is reset or discarded, the file must be uploaded again. For files used across multiple sessions, Google Drive can be **mounted** and the file can be read directly from it:

```
from google.colab import drive
drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/IT9002/sms_spam.csv")
```

The code above mounts the drive and accesses the dataset. Mounting is optional in this case, because the lab files are small enough to upload when needed.

## 5. Reading an Error

When an error occurs, the cell stops running and Python displays a traceback. The final line usually identifies the error type and describes its cause, so it is a useful place to begin reading. 

The cell below intentionally uses a **misspelled** column name and catches the resulting error so that its message can be displayed.

In [14]:
try:
    df["messsage"]
except KeyError as error:
    print("KeyError:", error)

KeyError: 'messsage'


A `KeyError` indicates that the requested column or key does not exist. In this example, the problem is the misspelled column name. Three error types are common in these workspaces:

- `NameError`: a variable is used **before the cell that defines it has been executed**.

- `KeyError`: a **requested column or key does not exist**, often because its name is incorrect.

- `FileNotFoundError`: the **data file is not available** at the specified location or has not been uploaded.

## Exercises

### Exercise 1: Long Messages

Identify and print the **5 longest messages** in the dataset, measured by characters, alongside their labels and lengths. 

`df["message"].str.len()` returns the length of each message, and `sort_values()` orders a DataFrame by a specified column.

In [15]:
# TODO: Identify and print the 5 longest messages in the dataset, measured by characters, alongside their labels and lengths.


**Expected result:** The longest message contains 910 characters, and all 5 of the longest messages are labeled as normal messages.

### Exercise 2: Counting Words

Count how many spam and normal messages contain the word `free`, regardless of letter case. 

`df["message"].str.contains("free", case=False)` identifies the rows in which the word occurs.

In [16]:
# TODO: mark the messages containing "free" in any casing, then print the count per label


**Expected result:** ~200 spam messages and ~70 normal messages contain the word. This corresponds to approximately 1/4 of spam messages and 1/70 of normal messages.